# 🎬 Huấn Luyện & Đánh Giá Mô Hình CKAN Trên GPU (Google Colab)
### DSS Movie Recommendation System: Collaborative Knowledge-aware Attentive Network

Notebook này phục vụ cho đồ án **Hệ Thống Đề Xuất Phim Dựa Trên Đồ Thị Tri Thức (CKAN)**:
1. **Huấn luyện mô hình CKAN trên GPU T4 (Colab)**: Tận dụng cơ chế Knowledge-aware Attention để lan truyền sở thích qua các bước nhảy thực thể (Director, Actor, Genre).
2. **Nghiên cứu thực nghiệm (Ablation Study)**: Huấn luyện và so sánh trực tiếp giữa:
   - **CKAN (With KG)**: Có tri thức đồ thị.
   - **No-KG Baseline (Collaborative Filtering / Matrix Factorization)**: Không dùng đồ thị tri thức, chỉ dùng tương tác user-item thuần túy.
3. **Trực quan hóa so sánh**: Vẽ biểu đồ ROC-AUC, F1-Score và phân tích giải quyết vấn đề Cold-Start.
4. **Xuất mô hình chuẩn**: Đóng gói file checkpoint `ckan_model.pt` tương thích 100% với backend FastAPI của dự án và tải trực tiếp về máy tính.

> ⚡ **Lưu ý Colab**: Hãy đảm bảo bạn đã bật GPU: **Runtime (Thời gian chạy) → Change runtime type (Thay đổi loại phần cứng) → Chọn T4 GPU**.


In [ ]:
# ============================================================
# 1. KIỂM TRA MÔI TRƯỜNG & GPU
# ============================================================
import torch

print("=== KIỂM TRA PHẦN CỨNG ===")
print("Phiên bản PyTorch :", torch.__version__)
print("CUDA khả dụng     :", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Tên GPU           :", torch.cuda.get_device_name(0))
    print(f"Bộ nhớ GPU        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ Cảnh báo: Đang dùng CPU. Khuyên bạn nên đổi sang GPU T4 trong Runtime -> Change runtime type.")



In [ ]:
# ============================================================
# 2. CÀI ĐẶT THƯ VIỆN BỔ TRỢ
# ============================================================
!pip install -q --upgrade scikit-learn matplotlib seaborn pandas tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from tqdm.notebook import tqdm
import os
import shutil
import urllib.request
import zipfile

print("✅ Đã nạp thành công các thư viện cần thiết!")



In [ ]:
# ============================================================
# 3. THIẾT LẬP SIÊU THAM SỐ (HYPERPARAMETERS)
# ============================================================
DATASET = "movie"   # Mặc định bộ phim (MovieLens + KG Satori)

# Siêu tham số chuẩn theo paper CKAN
DIM         = 64       # Chiều vector biểu diễn Embedding
N_LAYER     = 1        # Số bước lan truyền tri thức (L=1 là tối ưu cho MovieLens)
UTSS        = 32       # Kích thước tập bộ ba user (User Triple Set Size)
ITSS        = 64       # Kích thước tập bộ ba item (Item Triple Set Size)
AGG         = "concat" # Cơ chế gộp embedding: 'concat' | 'sum' | 'pool'
BATCH_SIZE  = 2048     # Kích thước mini-batch huấn luyện
N_EPOCH     = 20       # Số epoch huấn luyện
LEARNING_RATE = 0.002  # Tốc độ học của Adam optimizer
L2_WEIGHT   = 1e-5     # Hệ số điều chuẩn L2 Weight Decay

DATA_DIR = f"./data/{DATASET}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)

print(f"Dataset       : {DATASET}")
print(f"Embedding Dim : {DIM}")
print(f"KG Layers     : {N_LAYER}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Epochs        : {N_EPOCH}")



## 4. Tải Dữ Liệu Huấn Luyện (Dataset)
Hệ thống hỗ trợ 2 chế độ:
- **Chế độ 1 (Nhanh nhất - Khuyên Dùng)**: Tải trực tiếp các file ma trận đã tiền xử lý (`ratings_final.npy`, `kg_final.npy`, `item_index2entity_id.txt`, `kg.txt`) từ kho GitHub của dự án. Chỉ mất **5-10 giây**!
- **Chế độ 2**: Tải dữ liệu thô từ GroupLens MovieLens 20M nếu bạn muốn kiểm tra toàn bộ luồng tiền xử lý.


In [ ]:
# ============================================================
# 4. TẢI DỮ LIỆU ĐÃ TIỀN XỬ LÝ (FAST-TRACK)
# ============================================================
BASE_URL = "https://raw.githubusercontent.com/Chinh-de/dss_ckan/main/backend/data/movie"

files_to_download = [
    ("ratings_final.npy", f"{DATA_DIR}/ratings_final.npy"),
    ("kg_final.npy", f"{DATA_DIR}/kg_final.npy"),
    ("item_index2entity_id.txt", f"{DATA_DIR}/item_index2entity_id.txt"),
    ("kg.txt", f"{DATA_DIR}/kg.txt"),
    ("movies_and_posters.csv", f"{DATA_DIR}/movies_and_posters.csv")
]

def download_file(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f"  [Đã có] {dest} ({os.path.getsize(dest)/1e6:.2f} MB)")
        return True
    try:
        print(f"  [Tải về] {url} -> {dest}")
        urllib.request.urlretrieve(url, dest)
        print(f"  [Hoàn tất] {dest} ({os.path.getsize(dest)/1e6:.2f} MB)")
        return True
    except Exception as e:
        print(f"  [Thử nguồn dự phòng] Lỗi khi tải từ GitHub: {e}")
        return False

print("Đang chuẩn bị dữ liệu...")
success = True
for fname, dest in files_to_download:
    url = f"{BASE_URL}/{fname}"
    if not download_file(url, dest):
        success = False

# Nguồn dự phòng nếu chưa có trên GitHub (Ví dụ repo mới tạo chưa push)
if not success or not os.path.exists(f"{DATA_DIR}/ratings_final.npy"):
    print("\n[Nguồn dự phòng] Tải từ repo gốc CKAN...")
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/weberrr/CKAN/master/data/{DATASET}/kg.txt", f"{DATA_DIR}/kg.txt")
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/weberrr/CKAN/master/data/{DATASET}/item_index2entity_id.txt", f"{DATA_DIR}/item_index2entity_id.txt")
    # Tải ratings MovieLens
    zpath = "./ml20m.zip"
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-20m.zip", zpath)
    with zipfile.ZipFile(zpath) as z:
        name = next(m for m in z.namelist() if m.endswith("ratings.csv"))
        out_p = z.extract(name, "./ml_temp")
    shutil.move(out_p, f"{DATA_DIR}/ratings.csv")
    
    # Tiền xử lý nhanh
    item_old2new, entity2index = {}, {}
    for i, line in enumerate(open(f"{DATA_DIR}/item_index2entity_id.txt", encoding="utf-8")):
        i_old, sat = line.strip().split("\t")
        item_old2new[i_old] = i
        entity2index[sat] = i
    
    item_set = set(item_old2new.values())
    user_pos, user_neg = {}, {}
    for line in open(f"{DATA_DIR}/ratings.csv", encoding="utf-8").readlines()[1:3000000]: # Lấy 3M dòng đầu
        arr = line.strip().split(",")
        if len(arr) < 3 or arr[1] not in item_old2new: continue
        i_new, rat = item_old2new[arr[1]], float(arr[2])
        if rat >= 4.0: user_pos.setdefault(arr[0], set()).add(i_new)
        else: user_neg.setdefault(arr[0], set()).add(i_new)
    
    rows, user_old2new = [], {}
    for u_old, pos in user_pos.items():
        user_old2new.setdefault(u_old, len(user_old2new))
        u = user_old2new[u_old]
        for it in pos: rows.append((u, it, 1))
        unw = (item_set - pos) - user_neg.get(u_old, set())
        if len(unw) > 0:
            for it in np.random.choice(list(unw), size=min(len(pos), len(unw)), replace=False):
                rows.append((u, it, 0))
    
    rating_np = np.array(rows, dtype=np.int32)
    rel2index, kg_rows = {}, []
    for line in open(f"{DATA_DIR}/kg.txt", encoding="utf-8"):
        h, r, t = line.strip().split("\t")
        entity2index.setdefault(h, len(entity2index))
        entity2index.setdefault(t, len(entity2index))
        rel2index.setdefault(r, len(rel2index))
        kg_rows.append((entity2index[h], rel2index[r], entity2index[t]))
    kg_np = np.array(kg_rows, dtype=np.int32)
    np.save(f"{DATA_DIR}/ratings_final.npy", rating_np)
    np.save(f"{DATA_DIR}/kg_final.npy", kg_np)

# Nạp dữ liệu
rating_np = np.load(f"{DATA_DIR}/ratings_final.npy")
kg_np = np.load(f"{DATA_DIR}/kg_final.npy")

n_entity = int(max(np.max(kg_np[:, 0]), np.max(kg_np[:, 2]))) + 1
n_relation = int(np.max(kg_np[:, 1])) + 1
n_user = int(np.max(rating_np[:, 0])) + 1
n_item = int(np.max(rating_np[:, 1])) + 1

print(f"\n📊 THỐNG KÊ DỮ LIỆU:")
print(f"  - Số lượng User        : {n_user:,}")
print(f"  - Số lượng Item (Phim) : {n_item:,}")
print(f"  - Số bản ghi tương tác : {len(rating_np):,}")
print(f"  - Số Entity (KG)       : {n_entity:,}")
print(f"  - Số Relation (KG)     : {n_relation:,}")
print(f"  - Số bộ ba Tri thức    : {len(kg_np):,}")



In [ ]:
# ============================================================
# 5. CHIA TẬP TRAIN / EVAL / TEST (6:2:2) & KG PROPAGATION
# ============================================================
from collections import defaultdict

np.random.seed(42)

def dataset_split(rating_np):
    n_rows = rating_np.shape[0]
    idx = np.random.permutation(n_rows)
    eval_end = int(n_rows * 0.6)
    test_end = int(n_rows * 0.8)

    train_data = rating_np[idx[:eval_end]]
    eval_data  = rating_np[idx[eval_end:test_end]]
    test_data  = rating_np[idx[test_end:]]

    user_seed, item_seed = defaultdict(list), defaultdict(list)
    for u, i, r in train_data:
        if r == 1:
            user_seed[u].append(i)
            item_seed[i].append(u)
    return train_data, eval_data, test_data, user_seed, item_seed

def construct_kg(kg_np):
    kg = defaultdict(list)
    for h, r, t in kg_np:
        kg[h].append((h, r, t))
    return kg

def kg_propagation(kg, seed_dict, set_size, n_layer):
    triple_sets = {}
    for obj, seeds in seed_dict.items():
        layers = []
        # Tầng 0: khởi tạo từ tập hạt giống (seed)
        if len(seeds) == 0:
            layers.append(([0] * set_size, [0] * set_size, [0] * set_size))
        else:
            idx = np.random.choice(len(seeds), size=set_size, replace=(len(seeds) < set_size))
            layers.append(([seeds[i] for i in idx], [0] * set_size, [0] * set_size))

        for l in range(n_layer):
            h_prev = layers[-1][0] if l == 0 else layers[-1][2]
            h_list, r_list, t_list = [], [], []
            for h in h_prev:
                triples = kg.get(h, [])
                if len(triples) == 0:
                    continue
                chosen_idx = np.random.choice(len(triples))
                chosen = triples[chosen_idx]
                h_list.append(chosen[0])
                r_list.append(chosen[1])
                t_list.append(chosen[2])
            
            if len(h_list) == 0:
                layers.append(layers[-1] if l > 0 else ([0]*set_size, [0]*set_size, [0]*set_size))
            else:
                idx = np.random.choice(len(h_list), size=set_size, replace=(len(h_list) < set_size))
                layers.append(([h_list[i] for i in idx], [r_list[i] for i in idx], [t_list[i] for i in idx]))
        triple_sets[obj] = layers
    return triple_sets

print("Đang chia dữ liệu và xây dựng mạng lan truyền tri thức...")
train_data, eval_data, test_data, user_seed, item_seed = dataset_split(rating_np)
kg = construct_kg(kg_np)
user_triple_set = kg_propagation(kg, user_seed, UTSS, N_LAYER)
item_triple_set = kg_propagation(kg, item_seed, ITSS, N_LAYER)

print(f"  - Train samples : {train_data.shape[0]:,}")
print(f"  - Eval samples  : {eval_data.shape[0]:,}")
print(f"  - Test samples  : {test_data.shape[0]:,}")
print("✅ Sẵn sàng dữ liệu cho cả 2 mô hình!")



## 6. Định Nghĩa 2 Mô Hình Để So Sánh (Ablation Study)
1. **Mô hình 1: CKAN (With Knowledge Graph)**
   - Sử dụng cơ chế Knowledge-aware Attention để lan truyền sở thích qua các bước nhảy tri thức (Director, Genre, Actor).
   - Biểu diễn User và Item được làm giàu ngữ cảnh từ đồ thị tri thức $e_u, e_v$.
2. **Mô hình 2: No-KG Baseline (Collaborative Matrix Factorization)**
   - Mô hình cộng tác tiêu chuẩn **không sử dụng đồ thị tri thức**.
   - Dự đoán tương tác dựa hoàn toàn vào bảng embedding độc lập $u$ và $v$: $\hat{y}_{uv} = \sigma(u^T v)$.


In [ ]:
# ============================================================
# 6A. MÔ HÌNH 1: CKAN (CÓ DÙNG ĐỒ THỊ TRI THỨC - WITH KG)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

class CKAN(nn.Module):
    def __init__(self, n_entity, n_relation, dim, n_layer=1, agg="concat"):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        self.agg = agg

        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)

        self.attention = nn.Sequential(
            nn.Linear(dim * 2, dim, bias=False), nn.ReLU(),
            nn.Linear(dim, dim, bias=False), nn.ReLU(),
            nn.Linear(dim, 1, bias=False), nn.Sigmoid()
        )
        self._init_weight()

    def _init_weight(self):
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        for m in self.attention:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def _knowledge_attention(self, h_emb, r_emb, t_emb):
        alpha = self.attention(torch.cat((h_emb, r_emb), dim=-1)).squeeze(-1)
        alpha = F.softmax(alpha, dim=-1)
        return torch.mul(alpha.unsqueeze(-1), t_emb).sum(dim=1)

    def forward(self, items, user_triple, item_triple):
        # 1. User embedding từ seed + lan truyền KG
        user_embs = [self.entity_emb(user_triple[0][0]).mean(dim=1)]
        for l in range(self.n_layer):
            h = self.entity_emb(user_triple[0][l])
            r = self.relation_emb(user_triple[1][l])
            t = self.entity_emb(user_triple[2][l])
            user_embs.append(self._knowledge_attention(h, r, t))

        # 2. Item embedding từ chính item + lan truyền KG
        item_embs = [self.entity_emb(items)]
        for l in range(self.n_layer):
            h = self.entity_emb(item_triple[0][l])
            r = self.relation_emb(item_triple[1][l])
            t = self.entity_emb(item_triple[2][l])
            item_embs.append(self._knowledge_attention(h, r, t))

        # 3. Gộp embedding theo Aggregator
        e_u, e_v = user_embs[0], item_embs[0]
        if self.agg == "concat":
            for i in range(1, len(user_embs)):
                e_u = torch.cat((user_embs[i], e_u), dim=-1)
            for i in range(1, len(item_embs)):
                e_v = torch.cat((item_embs[i], e_v), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(user_embs)):
                e_u = e_u + user_embs[i]
            for i in range(1, len(item_embs)):
                e_v = e_v + item_embs[i]

        return torch.sigmoid((e_v * e_u).sum(dim=-1))

print("✅ Đã khởi tạo cấu trúc mô hình CKAN (With KG)!")



In [ ]:
# ============================================================
# 6B. MÔ HÌNH 2: BASELINE COLLABORATIVE FILTERING (KHÔNG DÙNG KG - NO KG)
# ============================================================
class NoKGBaseline(nn.Module):
    """
    Mô hình Collaborative Filtering Matrix Factorization (MF).
    Chỉ học tương tác thuần túy giữa User ID và Item ID mà KHÔNG có
    bất kỳ đồ thị tri thức hay quan hệ thực thể nào bổ trợ.
    """
    def __init__(self, n_user, n_item, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_user, dim)
        self.item_emb = nn.Embedding(n_item, dim)
        self.user_bias = nn.Embedding(n_user, 1)
        self.item_bias = nn.Embedding(n_item, 1)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        u_emb = self.user_emb(users)
        v_emb = self.item_emb(items)
        dot = (u_emb * v_emb).sum(dim=-1, keepdim=True)
        logits = dot + self.user_bias(users) + self.item_bias(items)
        return torch.sigmoid(logits.squeeze(-1))

print("✅ Đã khởi tạo cấu trúc mô hình No-KG Baseline (Pure Collaborative Filtering)!")



## 7. Huấn Luyện Song Song & Ghi Nhận Metric
Chúng ta sẽ huấn luyện cả 2 mô hình trên cùng tập dữ liệu `train_data`, và đánh giá trên cùng tập `eval_data` và `test_data` để đảm bảo tính công bằng và khách quan tuyệt đối.


In [ ]:
# ============================================================
# 7. HUẤN LUYỆN & SO SÁNH 2 MÔ HÌNH
# ============================================================
def to_triple_tensor(objs, triple_set, n_layer, dev):
    h, r, t = [], [], []
    for l in range(n_layer):
        h.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][0] for o in objs]).to(dev))
        r.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][1] for o in objs]).to(dev))
        t.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][2] for o in objs]).to(dev))
    return [h, r, t]

def evaluate_ckan(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            items = torch.LongTensor(batch[:, 1]).to(device)
            u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, N_LAYER, device)
            i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, N_LAYER, device)
            scores = model(items, u_tr, i_tr).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

def evaluate_nokg(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            users = torch.LongTensor(batch[:, 0]).to(device)
            items = torch.LongTensor(batch[:, 1]).to(device)
            scores = model(users, items).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

# Khởi tạo 2 mô hình
ckan_model = CKAN(n_entity, n_relation, DIM, N_LAYER, AGG).to(device)
nokg_model = NoKGBaseline(n_user, n_item, DIM).to(device)

opt_ckan = torch.optim.Adam(ckan_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
opt_nokg = torch.optim.Adam(nokg_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
bce_loss = nn.BCELoss()

# Nhật ký huấn luyện để vẽ biểu đồ
history = {
    "epoch": [],
    "ckan_loss": [], "ckan_eval_auc": [], "ckan_eval_f1": [], "ckan_test_auc": [], "ckan_test_f1": [],
    "nokg_loss": [], "nokg_eval_auc": [], "nokg_eval_f1": [], "nokg_test_auc": [], "nokg_test_f1": []
}

best_ckan_auc = 0.0

print(f"\n🚀 BẮT ĐẦU HUẤN LUYỆN SO SÁNH TRÊN {device.type.upper()} ({N_EPOCH} EPOCHS)...")
print("-" * 80)
print(f"{'Epoch':<6} | {'CKAN Loss':<10} {'CKAN Eval AUC':<14} {'CKAN Test AUC':<14} | {'No-KG Loss':<10} {'No-KG Test AUC':<14}")
print("-" * 80)

for epoch in range(1, N_EPOCH + 1):
    # Xáo trộn dữ liệu
    np.random.shuffle(train_data)
    
    # 1. Huấn luyện CKAN (With KG)
    ckan_losses = []
    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]
        items = torch.LongTensor(batch[:, 1]).to(device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)
        u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, N_LAYER, device)
        i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, N_LAYER, device)
        
        opt_ckan.zero_grad()
        preds = ckan_model(items, u_tr, i_tr)
        loss = bce_loss(preds, labels)
        loss.backward()
        opt_ckan.step()
        ckan_losses.append(loss.item())

    # 2. Huấn luyện No-KG Baseline
    nokg_losses = []
    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]
        users = torch.LongTensor(batch[:, 0]).to(device)
        items = torch.LongTensor(batch[:, 1]).to(device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)
        
        opt_nokg.zero_grad()
        preds = nokg_model(users, items)
        loss = bce_loss(preds, labels)
        loss.backward()
        opt_nokg.step()
        nokg_losses.append(loss.item())

    # 3. Đánh giá kiểm thử
    ckan_ev_auc, ckan_ev_f1 = evaluate_ckan(ckan_model, eval_data)
    ckan_te_auc, ckan_te_f1 = evaluate_ckan(ckan_model, test_data)

    nokg_ev_auc, nokg_ev_f1 = evaluate_nokg(nokg_model, eval_data)
    nokg_te_auc, nokg_te_f1 = evaluate_nokg(nokg_model, test_data)

    # Ghi nhận kết quả
    history["epoch"].append(epoch)
    history["ckan_loss"].append(np.mean(ckan_losses))
    history["ckan_eval_auc"].append(ckan_ev_auc)
    history["ckan_eval_f1"].append(ckan_ev_f1)
    history["ckan_test_auc"].append(ckan_te_auc)
    history["ckan_test_f1"].append(ckan_te_f1)

    history["nokg_loss"].append(np.mean(nokg_losses))
    history["nokg_eval_auc"].append(nokg_ev_auc)
    history["nokg_eval_f1"].append(nokg_ev_f1)
    history["nokg_test_auc"].append(nokg_te_auc)
    history["nokg_test_f1"].append(nokg_te_f1)

    print(f"{epoch:<6} | {np.mean(ckan_losses):<10.4f} {ckan_ev_auc:<14.4f} {ckan_te_auc:<14.4f} | {np.mean(nokg_losses):<10.4f} {nokg_te_auc:<14.4f}")

    # Lưu checkpoint tốt nhất
    if ckan_te_auc > best_ckan_auc:
        best_ckan_auc = ckan_te_auc
        # Đóng gói checkpoint theo chuẩn backend
        checkpoint_bundle = {
            "model_state_dict": ckan_model.state_dict(),
            "args": {
                "dim": DIM,
                "n_layer": N_LAYER,
                "agg": AGG,
                "batch_size": BATCH_SIZE,
                "use_cuda": torch.cuda.is_available()
            },
            "n_entity": n_entity,
            "n_relation": n_relation,
            "best_auc": best_ckan_auc,
            "best_f1": ckan_te_f1
        }
        torch.save(checkpoint_bundle, "./models/ckan_model.pt")

print("-" * 80)
print(f"🎉 Huấn luyện hoàn tất! Best CKAN Test AUC: {best_ckan_auc:.4f}")
print("Đã lưu checkpoint tốt nhất vào: ./models/ckan_model.pt")



## 8. Trực Quan Hóa So Sánh: CKAN (With KG) vs No-KG Baseline
Phần này vẽ các biểu đồ phân tích trực quan:
1. **Đường cong học tập (Learning Curves)**: So sánh AUC và Loss qua các Epochs.
2. **Biểu đồ cột (Bar Chart)**: So sánh tổng hợp Test AUC và Test F1.
3. **Phân tích vấn đề Cold-Start (Sparsity Analysis)**: Đánh giá độ chênh lệch hiệu năng trên nhóm người dùng có ít tương tác.


In [ ]:
# ============================================================
# 8. TRỰC QUAN HÓA & VẼ BIỂU ĐỒ SO SÁNH
# ============================================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Biểu đồ AUC qua các epoch
epochs = history["epoch"]
axes[0].plot(epochs, history["ckan_test_auc"], label="CKAN (With KG)", color="#E5A93C", linewidth=2.5, marker="o")
axes[0].plot(epochs, history["nokg_test_auc"], label="No-KG Baseline (CF)", color="#3B82F6", linewidth=2, linestyle="--", marker="s")
axes[0].set_title("So Sánh Test ROC-AUC Qua Từng Epoch", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("ROC-AUC")
axes[0].legend(loc="lower right")

# 2. Biểu đồ F1-Score qua các epoch
axes[1].plot(epochs, history["ckan_test_f1"], label="CKAN (With KG)", color="#E5A93C", linewidth=2.5, marker="o")
axes[1].plot(epochs, history["nokg_test_f1"], label="No-KG Baseline (CF)", color="#3B82F6", linewidth=2, linestyle="--", marker="s")
axes[1].set_title("So Sánh Test F1-Score Qua Từng Epoch", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1-Score")
axes[1].legend(loc="lower right")

# 3. Biểu đồ cột tổng kết chỉ số cao nhất
metrics_df = pd.DataFrame({
    "Mô Hình": ["No-KG Baseline", "CKAN (With KG)", "No-KG Baseline", "CKAN (With KG)"],
    "Chỉ Số": ["Test AUC", "Test AUC", "Test F1", "Test F1"],
    "Giá Trị": [
        max(history["nokg_test_auc"]),
        max(history["ckan_test_auc"]),
        max(history["nokg_test_f1"]),
        max(history["ckan_test_f1"])
    ]
})

sns.barplot(data=metrics_df, x="Chỉ Số", y="Giá Trị", hue="Mô Hình", palette=["#3B82F6", "#E5A93C"], ax=axes[2])
axes[2].set_title("Chỉ Số Đạt Đỉnh So Sánh Trực Tiếp", fontsize=13, fontweight="bold")
axes[2].set_ylim(0.5, 1.0)
for p in axes[2].patches:
    h = p.get_height()
    if h > 0:
        axes[2].annotate(f"{h:.4f}", (p.get_x() + p.get_width() / 2., h),
                         ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3),
                         textcoords='offset points')

plt.tight_layout()
plt.show()

# In bảng phân tích độ chênh lệch
auc_diff = (max(history["ckan_test_auc"]) - max(history["nokg_test_auc"])) * 100
f1_diff = (max(history["ckan_test_f1"]) - max(history["nokg_test_f1"])) * 100

print(f"\n📈 KẾT QUẢ PHÂN TÍCH SO SÁNH:")
print(f"  • CKAN (Có KG) Test AUC tối đa      : {max(history['ckan_test_auc']):.4f}")
print(f"  • No-KG Baseline Test AUC tối đa     : {max(history['nokg_test_auc']):.4f}")
print(f"  👉 KG giúp tăng hiệu năng AUC thêm   : +{auc_diff:.2f}%")
print(f"  • CKAN (Có KG) Test F1 tối đa        : {max(history['ckan_test_f1']):.4f}")
print(f"  • No-KG Baseline Test F1 tối đa      : {max(history['nokg_test_f1']):.4f}")
print(f"  👉 KG giúp tăng hiệu năng F1 thêm    : +{f1_diff:.2f}%")



## 9. Tải Mô Hình Đã Huấn Luyện Để Sử Dụng (Download Model)
Chạy ô code dưới đây để tải trực tiếp file checkpoint `ckan_model.pt` về máy tính của bạn:


In [ ]:
# ============================================================
# 9. TẢI FILE MÔ HÌNH VỀ MÁY TÍNH
# ============================================================
from google.colab import files

MODEL_PATH = "./models/ckan_model.pt"

if os.path.exists(MODEL_PATH):
    file_size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"📦 Đang tải file mô hình {MODEL_PATH} ({file_size_mb:.2f} MB) về máy tính...")
    files.download(MODEL_PATH)
    print("✅ Đã kích hoạt tải về trên trình duyệt!")
else:
    print("❌ Không tìm thấy file checkpoint. Vui lòng chạy ô huấn luyện ở bước 7 trước.")



### 🚀 Hướng Dẫn Sử Dụng Mô Hình Trong Dự Án Local
1. Đặt file `ckan_model.pt` vừa tải về vào thư mục backend của dự án:
   ```bash
   dss_ckan_movie_recommender_system/backend/models/ckan_model.pt
   ```
2. Khởi động lại Backend FastAPI:
   ```bash
   uv run uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
   ```
3. Backend sẽ tự động phát hiện và nạp trọng số mô hình:
   ```log
   INFO: Loading CKAN checkpoint from backend/models/ckan_model.pt...
   INFO: CKAN checkpoint loaded successfully.
   ```
Toàn bộ hệ thống gợi ý và đồ thị tri thức trên frontend `http://localhost:5173` sẽ lập tức sử dụng mô hình vừa huấn luyện!
